# Analysis of heads with diagonal patterns

The goal is to

In [ ]:
import math
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src" 

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

## See the heads of a model

In [ ]:
CACHE_PATH = Path("../data/cache/chatmode/qwen25_gqa_cache.pt")
cache = torch.load(CACHE_PATH, map_location="cpu")

meta = cache["meta"]
prompts = cache["prompts"]

print("Model:", cache["model_id"])
print("Q heads:", meta["num_q_heads"], "| KV heads:", meta["num_kv_heads"], "| KV groups:", meta["num_kv_groups"])

PROMPT_IDX = 1
LAYER_IDX = 0

MAX_LABEL_LEN = 18
FIGSIZE_PER_PANEL = 6

record = prompts[PROMPT_IDX]
tokens = record["tokens"]
attn = record["attentions"][LAYER_IDX]   # [bsz, q_heads, tgt_len, src_len]
assert attn is not None, "Attention weights are missing. Ensure attn_implementation='eager'."

attn = attn[0].float()                   # [q_heads, seq_len, seq_len]
num_heads, tgt_len, src_len = attn.shape
kv_map = record["layers"][LAYER_IDX]["kv_head_for_q"].tolist()

def short_tok(t: str, max_len: int = MAX_LABEL_LEN) -> str:
    t = t.replace("Ġ", "▁").replace("Ċ", "\\n")
    return t if len(t) <= max_len else t[: max_len - 1] + "…"

labels = [short_tok(t) for t in tokens]

ncols = 2
nrows = math.ceil(num_heads / ncols)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * FIGSIZE_PER_PANEL, nrows * FIGSIZE_PER_PANEL),
    squeeze=False
)

for h in range(nrows * ncols):
    ax = axes[h // ncols][h % ncols]
    if h >= num_heads:
        ax.axis("off")
        continue

    sns.heatmap(
        attn[h].numpy(),
        ax=ax,
        cmap="magma",
        vmin=0.0,
        vmax=float(attn.max()),
        cbar=False,
        square=True,
        xticklabels=labels,
        yticklabels=labels,
    )
    ax.set_title(f"Head q={h} → kv={kv_map[h]}")
    ax.set_xlabel("Source token")
    ax.set_ylabel("Target token")
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

fig.suptitle(
    f"Attention heatmaps | prompt={PROMPT_IDX} | layer={LAYER_IDX}\n"
    f"{record['text']}",
    y=1.02
)
plt.tight_layout()
plt.show()

## Explore one head deeply

### View input/outputs

In [ ]:
CACHE_PATH = Path("../data/cache/chatmode/qwen25_full_block_cache.pt")

prompt_id = 1
layer_idx = 0
head_idx = 11
batch_idx = 0
max_tokens = 48

payload = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
rec = payload["prompts"][prompt_id]
layer = rec["layers"][layer_idx]
meta = payload["meta"]

num_q_heads = meta["num_q_heads"]       # 14
num_kv_heads = meta["num_kv_heads"]     # 2
num_kv_groups = meta["num_kv_groups"]   # 7
head_dim = meta["head_dim"]             # 64
hidden_size = layer["block_input"].shape[-1]  # 896

kv_head_idx = int(layer["kv_head_for_q"][head_idx].item())
tokens = rec["tokens"][:max_tokens]
tokens = [short_tok(t) for t in tokens]
T = len(tokens)

# Colonnes dans l'espace d_model qui appartiennent à cette tête q
q_col_start = head_idx * head_dim
q_col_end   = q_col_start + head_dim

# Colonnes pour la tête KV associée
kv_col_start = kv_head_idx * head_dim
kv_col_end   = kv_col_start + head_dim

def get(key, sub_head=None):
    t = layer[key][batch_idx, :max_tokens].float()
    if sub_head is not None:
        t = t[:, sub_head]
    return t

def highlight_cols(ax, start, end, nrows, label=None, color="cyan", lw=2):
    rect = mpatches.FancyArrowPatch(
        (start, 0), (end, 0),
        arrowstyle='-', color=color, lw=0,
    )
    rect2 = mpatches.Rectangle(
        (start, 0), end - start, nrows,
        linewidth=lw, edgecolor=color, facecolor="none",
        transform=ax.transData, clip_on=False,
        label=label or f"head {head_idx} cols [{start}:{end}]"
    )
    ax.add_patch(rect2)

def heatmap(ax, data, title, cmap="coolwarm", center=0,
            xticklabels=False, yticklabels=None, col_range=None):
    sns.heatmap(
        data, ax=ax, cmap=cmap, center=center, cbar=True,
        xticklabels=xticklabels,
        yticklabels=yticklabels if yticklabels is not None else False,
    )
    ax.set_title(title, fontsize=11)
    if col_range is not None:
        highlight_cols(ax, col_range[0], col_range[1], data.shape[0],
                       label=f"head {head_idx}")
        ax.legend(loc="upper right", fontsize=8, framealpha=0.6)

# ---------- données ----------
attn_weights = layer["attn_weights"][batch_idx, head_idx, :max_tokens, :max_tokens].float()

# entrée attention (full d_model) — highlight des colonnes de Q
block_in_full = get("block_input")                                   # [T, 896]
norm1_out_full = get("input_layernorm_out")                         # [T, 896]

# V : dim head_dim (64)
v_mat = get("v_pre", sub_head=kv_head_idx)                          # [T, 64]

# sortie par tête (head_dim)
head_out = get("head_out", sub_head=head_idx)                       # [T, 64]

# résiduel + full d_model — highlight des colonnes
resid1_full = get("residual_after_attn")                            # [T, 896]
norm2_out_full = get("post_attn_layernorm_out")                     # [T, 896]
mlp_out_full = get("mlp_out")                                       # [T, 896]
final_out_full = get("block_output")                                # [T, 896]

# ---------- figure ----------
rows = [
    # (data, title, col_range à highlighter, cmap, center)
    (attn_weights,   f"Attention — head {head_idx}",                        None,           "viridis", None),
    (block_in_full,  "Entrée du bloc (d_model=896)",                        (q_col_start, q_col_end), "coolwarm", 0),
    (norm1_out_full, "Après input_layernorm (d_model=896)",                 (q_col_start, q_col_end), "coolwarm", 0),
    (v_mat,          f"Matrice V — kv head {kv_head_idx} (head_dim=64)",    None,           "coolwarm", 0),
    (head_out,       f"Sortie après attention — head {head_idx} (head_dim=64)", None,        "coolwarm", 0),
    (resid1_full,    "Après 1er résiduel (d_model=896)",                    (q_col_start, q_col_end), "coolwarm", 0),
    (norm2_out_full, "Après post_attention_layernorm (d_model=896)",        None,           "coolwarm", 0),
    (mlp_out_full,   "Sortie MLP (d_model=896)",                            None,           "coolwarm", 0),
    (final_out_full, "Après 2e résiduel / sortie du bloc (d_model=896)",    None,           "coolwarm", 0),
]

fig, axes = plt.subplots(len(rows), 1, figsize=(24, 7 * len(rows)), constrained_layout=True)
fig.suptitle(
    f"Prompt {prompt_id} | Layer {layer_idx} | Q-head {head_idx} "
    f"(KV head {kv_head_idx})  —  {T} tokens",
    fontsize=13, fontweight="bold"
)

for ax, (data, title, col_range, cmap, center) in zip(axes, rows):
    ytl = tokens if data.shape[0] == T else False
    heatmap(ax, data.numpy(), title, cmap=cmap, center=center,
            yticklabels=ytl, col_range=col_range)
    ax.set_xlabel("dim →")
    ax.set_ylabel("tokens →")

plt.show()

### Pos - sym scores

In [ ]:
from gemma_explore.qwen_scores import *

bundle = load_qwen_bundle()
scores = collect_last_token_attentions(
    bundle,
    prompt="I'm testing attention heads in a small transformer model.",
    n_blocks=8,
)

plot_head_score_heatmap(scores.mean_scores, score_type="sym")

In [ ]:
freq = collect_frequency_scores(
    bundle,
    prompt="I'm testing attention heads in a small transformer model.",
    n_blocks=8,
)

plot_frequency_scores(freq, layer_idx=0, head_idx=1)
plot_frequency_logit_norms(freq, layer_idx=0, head_idx=1)